# Notebook 02 — Training Loop

This notebook runs the ablation study training loop across all four conditions.

Each condition is trained for a fixed number of epochs with a fixed set of
random seeds. A separate model is instantiated and trained for each
condition–seed combination. Checkpoints and per-seed F1 scores are written
to `/kaggle/working/` and saved to the consolidated dataset at the end of
the session.

The four conditions are:

| Condition | KG source | Fusion type |
|---|---|---|
| A_late | Phase A (Wikidata) | Late fusion |
| A_additive | Phase A (Wikidata) | Additive fusion |
| B_late | Phase B (Parliamentary) | Late fusion |
| B_additive | Phase B (Parliamentary) | Additive fusion |

A no-KG baseline condition (`no_kg`) is also included, passing zero vectors
for all tokens, to establish a within-notebook reference point against the
Adkins et al. 2025 reported F1 of 0.7652.

In [1]:
import torch                                              # tensor operations and model framework
import torch.nn as nn                                     # neural network layers and loss functions
from torch.optim import AdamW                             # optimiser; imported from torch not transformers
from torch.utils.data import DataLoader, TensorDataset   # batching and dataset wrapping
from transformers import (                                # gaBERT tokeniser, encoder, and LR scheduler
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)
import pickle                                             # loading pre-computed KG embedding pickles
import numpy as np                                        # array handling and seed control
import pandas as pd                                       # entity CSV loading for surface_to_qid
import json                                               # serialising F1 score results to disk
import os                                                 # path existence checks and directory creation
import random                                             # Python-level seed control
import subprocess                                         # runtime pip install for torchcrf
subprocess.run(['pip', 'install', 'pytorch-crf', '-q'])  # torchcrf not pre-installed on Kaggle
from torchcrf import CRF                                  # conditional random field decode and loss

In [2]:
# Constants and paths

DATA       = '/kaggle/input/datasets/michaelmarkey64/irish-ner-kg-consolidated'
WORKING    = '/kaggle/working'
MODEL_NAME = 'DCU-NLP/bert-base-irish-cased-v1'
MAX_LEN    = 128
BERT_DIM   = 768
KG_DIM     = 128

EPOCHS      = 5
BATCH_SIZE  = 16
LR          = 2e-5
SEEDS       = [42, 123, 256, 512, 999, 1024, 2048]

ABLATION_CONDITIONS = {
    'no_kg':       {'kg_source': None,      'fusion_type': 'late'},
    'A_late':      {'kg_source': 'phase_a', 'fusion_type': 'late'},
    'A_additive':  {'kg_source': 'phase_a', 'fusion_type': 'additive'},
    'B_late':      {'kg_source': 'phase_b', 'fusion_type': 'late'},
    'B_additive':  {'kg_source': 'phase_b', 'fusion_type': 'additive'},
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Conditions: {list(ABLATION_CONDITIONS.keys())}')
print(f'Seeds: {SEEDS}')
print(f'Epochs: {EPOCHS}, batch size: {BATCH_SIZE}, lr: {LR}')

Device: cpu
Conditions: ['no_kg', 'A_late', 'A_additive', 'B_late', 'B_additive']
Seeds: [42, 123, 256, 512, 999, 1024, 2048]
Epochs: 5, batch size: 16, lr: 2e-05


In [3]:
# CELL 4 — Load KG embeddings and reconstruct surface_to_qid from node CSVs

with open(f'{DATA}/kg/phase_a/TransE_phase_a_embeddings.pkl', 'rb') as f:
    emb_a = pickle.load(f)                           # flat dict {QID: array}

with open(f'{DATA}/kg/phase_b/TransE_phase_b_embeddings.pkl', 'rb') as f:
    emb_b = pickle.load(f)                           # nested dict with by_surface and by_qid

emb_b_surface = emb_b['by_surface']                 # direct surface string lookup for Phase B
emb_b_qid     = emb_b['by_qid']                     # QID lookup for Phase B (not used in training)

# Reconstruct surface_to_qid from Phase A node CSVs
# Not saved to consolidated dataset; rebuilt here to avoid external dependency
per_nodes  = pd.read_csv(f'{DATA}/kg/phase_a/per_nodes.csv')
org_nodes  = pd.read_csv(f'{DATA}/kg/phase_a/org_nodes.csv')
loc_nodes  = pd.read_csv(f'{DATA}/kg/phase_a/loc_nodes.csv')
stub_nodes = pd.read_csv(f'{DATA}/kg/phase_a/stub_nodes.csv')

surface_to_qid = {}

def register_surfaces(df, surface_cols):
    # Register all non-null surface string variants as keys mapping to QID
    for col in surface_cols:
        if col in df.columns:
            for _, row in df.iterrows():
                if pd.notna(row[col]) and str(row[col]).strip():
                    surface_to_qid[str(row[col]).strip()] = row['qid']

register_surfaces(per_nodes,  ['label_en', 'label_ga', 'full_name_en'])
register_surfaces(org_nodes,  ['label_en', 'label_ga'])
register_surfaces(loc_nodes,  ['label_en', 'label_ga'])
register_surfaces(stub_nodes, ['label_en', 'label_ga'])

print(f'Phase A — {len(emb_a)} QID entries')
print(f'Phase B — {len(emb_b_surface)} surface entries, {len(emb_b_qid)} QID entries')
print(f'surface_to_qid — {len(surface_to_qid)} entries reconstructed')

Phase A — 1563 QID entries
Phase B — 575 surface entries, 180 QID entries
surface_to_qid — 2086 entries reconstructed


In [4]:
# CELL 5 — KG vector lookup functions and CoNLL loader

def get_kg_vector_a(surface, emb_a, surface_to_qid, dim):
    # Phase A: two-step lookup — surface string -> QID -> TransE vector
    # Returns zero vector of shape (dim,) on miss at either step
    qid = surface_to_qid.get(surface)
    if qid is None:
        return np.zeros(dim, dtype=np.float32)
    vec = emb_a.get(qid)
    if vec is None:
        return np.zeros(dim, dtype=np.float32)
    return vec.astype(np.float32)

def get_kg_vector_b(surface, emb_b_surface, dim):
    # Phase B: direct surface string lookup via by_surface sub-dict
    # Returns zero vector of shape (dim,) on miss
    vec = emb_b_surface.get(surface)
    if vec is None:
        return np.zeros(dim, dtype=np.float32)
    return vec.astype(np.float32)

def load_conll(path):
    # Load a CoNLL file into parallel token and label lists
    # Adapted from read_conll_file() in 02_NER_Training.ipynb
    tokens_list, labels_list = [], []
    tokens, labels = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line == '':
                if tokens:
                    tokens_list.append(tokens)
                    labels_list.append(labels)
                    tokens, labels = [], []
            else:
                parts = line.split()
                tokens.append(parts[0])
                labels.append(parts[-1])
    if tokens:
        tokens_list.append(tokens)
        labels_list.append(labels)
    return tokens_list, labels_list

train_tokens, train_labels = load_conll(f'{DATA}/conll/train_final.conll')
val_tokens,   val_labels   = load_conll(f'{DATA}/conll/NER_Irish_validation.conll')
test_tokens,  test_labels  = load_conll(f'{DATA}/conll/NER_Irish_test.conll')

print(f'Train sentences: {len(train_tokens)}')
print(f'Val sentences:   {len(val_tokens)}')
print(f'Test sentences:  {len(test_tokens)}')

Train sentences: 1006
Val sentences:   100
Test sentences:  140


In [5]:
# CELL 6 — Tokeniser, label vocabulary, encoding and KG vector construction

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Build label vocabulary from training labels
all_labels = sorted({lab for seq in train_labels for lab in seq})
label2id   = {lab: i for i, lab in enumerate(all_labels)}
id2label   = {i: lab for lab, i in label2id.items()}
NUM_LABELS = len(label2id)
print(f'Label vocabulary ({NUM_LABELS} classes): {all_labels}')

def encode_split(tokens_list, labels_list, tokenizer, max_len, label2id, kg_source, dim):
    # Tokenise sentences, align labels to wordpiece tokens, and construct per-token KG vectors
    # kg_source: 'phase_a', 'phase_b', or None (zero vectors throughout)
    # Returns input_ids, attention_mask, label_ids, kg_vectors as stacked tensors
    input_ids_list, mask_list, label_ids_list, kg_list = [], [], [], []

    for tokens, labels in zip(tokens_list, labels_list):
        enc_tokens, enc_labels, enc_kg = [], [], []

        for token, label in zip(tokens, labels):
            sub = tokenizer.tokenize(token)
            if not sub:
                continue
            enc_tokens.extend(sub)
            enc_labels.extend([label2id[label]] + [-100] * (len(sub) - 1))

            # Assign KG vector to first subword token; zero vector to continuations
            if kg_source == 'phase_a':
                vec = get_kg_vector_a(token, emb_a, surface_to_qid, dim)
            elif kg_source == 'phase_b':
                vec = get_kg_vector_b(token, emb_b_surface, dim)
            else:
                vec = np.zeros(dim, dtype=np.float32)   # no_kg condition

            enc_kg.extend([vec] + [np.zeros(dim, dtype=np.float32)] * (len(sub) - 1))

        # Truncate to max_len - 2, then add CLS and SEP
        enc_tokens = enc_tokens[:max_len - 2]
        enc_labels = enc_labels[:max_len - 2]
        enc_kg     = enc_kg[:max_len - 2]

        ids  = tokenizer.convert_tokens_to_ids(['[CLS]'] + enc_tokens + ['[SEP]'])
        labs = [-100] + enc_labels + [-100]
        kgs  = [np.zeros(dim, dtype=np.float32)] + enc_kg + [np.zeros(dim, dtype=np.float32)]

        # Pad to max_len
        pad_len = max_len - len(ids)
        mask    = [1] * len(ids) + [0] * pad_len
        ids     = ids  + [0] * pad_len
        labs    = labs + [-100] * pad_len
        kgs     = kgs  + [np.zeros(dim, dtype=np.float32)] * pad_len

        input_ids_list.append(torch.tensor(ids,  dtype=torch.long))
        mask_list.append(     torch.tensor(mask, dtype=torch.long))
        label_ids_list.append(torch.tensor(labs, dtype=torch.long))
        kg_list.append(       torch.tensor(np.stack(kgs), dtype=torch.float32))

    return (
        torch.stack(input_ids_list),
        torch.stack(mask_list),
        torch.stack(label_ids_list),
        torch.stack(kg_list)
    )

# Encode all splits for each KG source upfront to avoid re-encoding inside the training loop
encoded = {}
for condition, config in ABLATION_CONDITIONS.items():
    src = config['kg_source']
    if src not in encoded:                              # reuse encoding if source already done
        encoded[src] = {
            'train': encode_split(train_tokens, train_labels, tokenizer, MAX_LEN, label2id, src, KG_DIM),
            'val':   encode_split(val_tokens,   val_labels,   tokenizer, MAX_LEN, label2id, src, KG_DIM),
            'test':  encode_split(test_tokens,  test_labels,  tokenizer, MAX_LEN, label2id, src, KG_DIM),
        }
        ids, mask, labs, kgs = encoded[src]['train']
        print(f'Encoded kg_source={src}: input_ids={ids.shape}, kg_vectors={kgs.shape}')

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Label vocabulary (7 classes): ['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
Encoded kg_source=None: input_ids=torch.Size([1006, 128]), kg_vectors=torch.Size([1006, 128, 128])
Encoded kg_source=phase_a: input_ids=torch.Size([1006, 128]), kg_vectors=torch.Size([1006, 128, 128])
Encoded kg_source=phase_b: input_ids=torch.Size([1006, 128]), kg_vectors=torch.Size([1006, 128, 128])


In [6]:
# CELL 7 — GaBERTCRF model class

class GaBERTCRF(nn.Module):
    # Configurable gaBERT-CRF model supporting late fusion and additive fusion.
    # fusion_type 'late':     last 4 BERT layers concatenated + KG vector appended (3200-dim -> classifier)
    # fusion_type 'additive': KG vector projected to 768-dim and added elementwise to final hidden state

    def __init__(self, model_name, num_labels, fusion_type='late', kg_dim=128):
        super().__init__()
        self.fusion_type = fusion_type
        self.num_labels  = num_labels
        self.kg_dim      = kg_dim

        self.bert = AutoModel.from_pretrained(model_name, output_hidden_states=True)

        if fusion_type == 'late':
            classifier_input_dim = 4 * BERT_DIM + kg_dim   # 3072 + 128 = 3200
        elif fusion_type == 'additive':
            classifier_input_dim = BERT_DIM                 # 768; KG projected and added elementwise
            self.kg_projection = nn.Linear(kg_dim, BERT_DIM)
        else:
            raise ValueError(f"fusion_type must be 'late' or 'additive', got '{fusion_type}'")

        self.classifier = nn.Linear(classifier_input_dim, num_labels)
        self.crf        = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, kg_vectors, labels=None):
        outputs       = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.hidden_states

        if self.fusion_type == 'late':
            bert_repr = torch.cat(hidden_states[-4:], dim=-1)
            combined  = torch.cat([bert_repr, kg_vectors], dim=-1)
        elif self.fusion_type == 'additive':
            bert_repr    = hidden_states[-1]
            kg_projected = self.kg_projection(kg_vectors)
            combined     = bert_repr + kg_projected

        emissions = self.classifier(combined)

        if labels is not None:
            crf_mask = attention_mask.bool()     # True for all real tokens including CLS/SEP
            labels_crf = labels.clone()
            labels_crf[labels_crf == -100] = 0  # replace ignored positions with a valid index
            loss = -self.crf(emissions, labels_crf, mask=crf_mask, reduction='mean')
            return loss
        else:
            mask  = attention_mask.bool()
            preds = self.crf.decode(emissions, mask=mask)
            return preds

print('GaBERTCRF defined')

GaBERTCRF defined


In [7]:
# CELL 8 — Evaluation function

def evaluate(model, input_ids, attention_mask, labels, kg_vectors, batch_size, device, id2label):
    # Run inference over a full split and return token-level F1 (excluding -100 positions)
    # Returns a dict with per-class and macro F1 scores
    model.eval()
    dataset    = TensorDataset(input_ids, attention_mask, labels, kg_vectors)
    loader     = DataLoader(dataset, batch_size=batch_size)

    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            b_ids, b_mask, b_labs, b_kg = [x.to(device) for x in batch]
            preds = model(b_ids, b_mask, b_kg)             # list of lists, one per sentence

            for i, pred_seq in enumerate(preds):
                lab_seq = b_labs[i].tolist()
                for pred, lab in zip(pred_seq, lab_seq):
                    if lab == -100:                         # skip padding and continuation tokens
                        continue
                    all_preds.append(id2label[pred])
                    all_labels.append(id2label[lab])

    # Per-class counts for precision, recall, F1
    classes   = sorted(set(all_labels))
    results   = {}
    f1_scores = []

    for cls in classes:
        tp = sum(p == cls and l == cls for p, l in zip(all_preds, all_labels))
        fp = sum(p == cls and l != cls for p, l in zip(all_preds, all_labels))
        fn = sum(p != cls and l == cls for p, l in zip(all_preds, all_labels))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        results[cls] = {'precision': precision, 'recall': recall, 'f1': f1}
        if cls != 'O':                                      # exclude O from macro average
            f1_scores.append(f1)

    results['macro_f1'] = np.mean(f1_scores) if f1_scores else 0.0
    return results

print('evaluate() defined')

evaluate() defined


In [8]:
SEEDS  = [42, 123, 256, 512, 999, 1024, 2048]
EPOCHS = 5
ABLATION_CONDITIONS = {
    'no_kg':       {'kg_source': None,      'fusion_type': 'late'},
    'A_late':      {'kg_source': 'phase_a', 'fusion_type': 'late'},
    'A_additive':  {'kg_source': 'phase_a', 'fusion_type': 'additive'},
    'B_late':      {'kg_source': 'phase_b', 'fusion_type': 'late'},
    'B_additive':  {'kg_source': 'phase_b', 'fusion_type': 'additive'},
}

In [9]:
# CELL 9 — Training loop across all conditions and seeds

# BYPASS TOGGLE: Set to False because we train on Colab and import precomputed results.
TRAIN_MODELS = False  

def set_seed(seed):
    # Fix all random state for reproducibility across torch, numpy, and Python
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

if TRAIN_MODELS:
    results_all = {}   # {condition: {seed: {'val_f1': ..., 'test_f1': ...}}}

    for condition, config in ABLATION_CONDITIONS.items():
        src          = config['kg_source']
        fusion_type  = config['fusion_type']
        results_all[condition] = {}

        train_ids, train_mask, train_lab, train_kg = encoded[src]['train']
        val_ids,   val_mask,   val_lab,   val_kg   = encoded[src]['val']
        test_ids,  test_mask,  test_lab,  test_kg  = encoded[src]['test']

        train_dataset = TensorDataset(train_ids, train_mask, train_lab, train_kg)

        print(f'\n--- Condition: {condition} ---')

        for seed in SEEDS:
            set_seed(seed)

            model     = GaBERTCRF(MODEL_NAME, NUM_LABELS, fusion_type=fusion_type, kg_dim=KG_DIM).to(device)
            optimiser = AdamW(model.parameters(), lr=LR)
            loader    = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            total_steps    = len(loader) * EPOCHS
            warmup_steps   = int(0.1 * total_steps)           # 10% warmup
            scheduler = get_linear_schedule_with_warmup(optimiser, warmup_steps, total_steps)

            for epoch in range(EPOCHS):
                model.train()
                epoch_loss = 0.0
                for batch in loader:
                    b_ids, b_mask, b_labs, b_kg = [x.to(device) for x in batch]
                    optimiser.zero_grad()
                    loss = model(b_ids, b_mask, b_kg, labels=b_labs)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # gradient clipping
                    optimiser.step()
                    scheduler.step()
                    epoch_loss += loss.item()
                print(f'  seed={seed} epoch={epoch+1}/{EPOCHS} loss={epoch_loss/len(loader):.4f}')

            val_results  = evaluate(model, val_ids,  val_mask,  val_lab,  val_kg,  BATCH_SIZE, device, id2label)
            test_results = evaluate(model, test_ids, test_mask, test_lab, test_kg, BATCH_SIZE, device, id2label)

            val_f1  = val_results['macro_f1']
            test_f1 = test_results['macro_f1']
            results_all[condition][seed] = {'val_f1': val_f1, 'test_f1': test_f1}
            print(f'  seed={seed} val_macro_f1={val_f1:.4f} test_macro_f1={test_f1:.4f}')

            # Save checkpoint
            ckpt_path = f'{WORKING}/{condition}_seed{seed}.pt'
            torch.save(model.state_dict(), ckpt_path)

            del model                                           # free memory before next seed

    # Persist results to disk
    results_path = f'{WORKING}/ablation_results.json'
    with open(results_path, 'w') as f:
        json.dump({c: {str(s): v for s, v in sv.items()} for c, sv in results_all.items()}, f, indent=2)

    print(f'\nResults written to {results_path}')

else:
    print("Bypassing heavy training loop loop. Loading precomputed results via attached dataset.")

Bypassing heavy training loop loop. Loading precomputed results via attached dataset.


In [15]:
# CELL 10 — Results summary

import os
import json
import numpy as np
import pandas as pd

# The exact path Kaggle found
kaggle_dataset_path = '/kaggle/input/datasets/michaelmarkey64/irish-ner-ablation-results/ablation_results.json'

if os.path.exists(kaggle_dataset_path):
    path = kaggle_dataset_path
else:
    path = f'{WORKING}/ablation_results.json'

print(f"Loading precomputed results from: {path}")

with open(path) as f:
    results_all = json.load(f)

rows = []
for condition, seed_results in results_all.items():
    val_f1s  = [v['val_f1']  for v in seed_results.values()]
    test_f1s = [v['test_f1'] for v in seed_results.values()]
    rows.append({
        'condition':       condition,
        'n_seeds':         len(seed_results),
        'val_f1_mean':     np.mean(val_f1s),
        'val_f1_std':      np.std(val_f1s),
        'test_f1_mean':    np.mean(test_f1s),
        'test_f1_std':     np.std(test_f1s),
    })

summary = pd.DataFrame(rows).sort_values('test_f1_mean', ascending=False)
print("\n--- ABLATION RESULTS SUMMARY ---")
print(summary.to_string(index=False, float_format='{:.4f}'.format))

Loading precomputed results from: /kaggle/input/datasets/michaelmarkey64/irish-ner-ablation-results/ablation_results.json

--- ABLATION RESULTS SUMMARY ---
 condition  n_seeds  val_f1_mean  val_f1_std  test_f1_mean  test_f1_std
    A_late        7       0.8306      0.0082        0.8265       0.0075
     no_kg        7       0.8312      0.0088        0.8264       0.0064
    B_late        7       0.8297      0.0089        0.8254       0.0071
B_additive        7       0.8287      0.0077        0.8224       0.0064
A_additive        7       0.8296      0.0087        0.8205       0.0076


In [16]:
# CELL 11 — Confirm results written to working directory

import os
import json

results_path = '/kaggle/input/datasets/michaelmarkey64/irish-ner-ablation-results/ablation_results.json'

if os.path.exists(results_path):
    with open(results_path) as f:
        saved = json.load(f)
    print(f'ablation_results.json confirmed at attached dataset input path: {results_path}')
    for condition, seed_results in saved.items():
        for seed, scores in seed_results.items():
            print(f'  {condition} seed={seed}: val_f1={scores["val_f1"]:.4f} test_f1={scores["test_f1"]:.4f}')
else:
    results_path_fallback = f'{WORKING}/ablation_results.json'
    if os.path.exists(results_path_fallback):
        with open(results_path_fallback) as f:
            saved = json.load(f)
        print(f'ablation_results.json confirmed at fallback working path: {results_path_fallback}')
        for condition, seed_results in saved.items():
            for seed, scores in seed_results.items():
                print(f'  {condition} seed={seed}: val_f1={scores["val_f1"]:.4f} test_f1={scores["test_f1"]:.4f}')
    else:
        print('WARNING: ablation_results.json not found in attached input dataset or /kaggle/working/')

ablation_results.json confirmed at attached dataset input path: /kaggle/input/datasets/michaelmarkey64/irish-ner-ablation-results/ablation_results.json
  no_kg seed=42: val_f1=0.8394 test_f1=0.8156
  no_kg seed=123: val_f1=0.8366 test_f1=0.8206
  no_kg seed=256: val_f1=0.8293 test_f1=0.8250
  no_kg seed=512: val_f1=0.8366 test_f1=0.8272
  no_kg seed=999: val_f1=0.8223 test_f1=0.8274
  no_kg seed=1024: val_f1=0.8148 test_f1=0.8349
  no_kg seed=2048: val_f1=0.8391 test_f1=0.8341
  A_late seed=42: val_f1=0.8381 test_f1=0.8158
  A_late seed=123: val_f1=0.8285 test_f1=0.8182
  A_late seed=256: val_f1=0.8393 test_f1=0.8250
  A_late seed=512: val_f1=0.8318 test_f1=0.8235
  A_late seed=999: val_f1=0.8245 test_f1=0.8308
  A_late seed=1024: val_f1=0.8148 test_f1=0.8374
  A_late seed=2048: val_f1=0.8377 test_f1=0.8346
  A_additive seed=42: val_f1=0.8330 test_f1=0.8106
  A_additive seed=123: val_f1=0.8123 test_f1=0.8188
  A_additive seed=256: val_f1=0.8419 test_f1=0.8128
  A_additive seed=512: val

## Results — Ablation Study (Production)

The table above reports macro F1 across all conditions and seeds. Results in this notebook reflect the **final production run across 5 epochs and 7 random seeds**. These values represent the finalized empirical foundation for dissertation reporting and directly reflect true model performance.

### What the table shows

Each row corresponds to one ablation condition (KG source $\times$ fusion type). The `no_kg` baseline passes zero vectors for all tokens and serves as an within-notebook reference against the Adkins et al. 2025 reported F1 of 0.7652. Notably, our baseline architecture achieves a mean test F1 of 0.8264, substantially exceeding the 0.7652 reference mark.

The four KG conditions form a 2×2 factorial design:

* **KG source effect**: A_late vs B_late and A_additive vs B_additive isolate the contribution of KG domain (Wikidata vs parliamentary co-occurrence) with fusion architecture held constant.
* **Fusion architecture effect**: A_late vs A_additive and B_late vs B_additive isolate the contribution of fusion type with KG source held constant. At a macro level, late fusion architectures ($\text{mean F1} \approx 0.825 \text{--} 0.826$) consistently hold their ground or slightly lead, while additive projection configurations suffer a visible performance drop ($\text{mean F1} \approx 0.820 \text{--} 0.822$).
* **Interaction**: whether any domain advantage is consistent across both fusion types is the primary dissertation claim, tested via two-sample Wilcoxon signed-rank in Notebook 03.

Statistical analysis and formal significance interpretation are deferred to Notebook 03.